# Experiment 4 — two-layer model

The first layer estimates the monthly market level from aggregate time-series features. The second layer estimates structural rent from location and flat attributes after removing time-dependent variables. Validation is chronological: validation targets are never used to build either layer.

**Executed result:** At listing level, the structural model scored RMSE **548.904**, the time-only model **674.200**, and the tuned blend **538.165** (MAE 437.511, R² 0.3031). The optimizer assigned only **0.0607** weight to the time component and 0.9393 to the structural component. The separate monthly time-model RMSE of 174.253 is an aggregate-series metric and is not comparable with listing-level RMSE. The blend improved on either component in this experiment but remained substantially weaker than the later direct tabular models.

In [1]:
from pathlib import Path
import os
import sys
import warnings

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'), Path('/content/drive/MyDrive/KDDM-project')]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import load_datasets, preprocess_frame
data_dir, datasets = load_datasets(project_root / 'data')
train_raw, test_raw = datasets['train'], datasets['test']
train_df = preprocess_frame(train_raw, datasets)
test_df = preprocess_frame(test_raw, datasets)
train_df['period'] = pd.to_datetime(train_df['RENT_APPROVAL_DATE'] + '-01')
test_df['period'] = pd.to_datetime(test_df['RENT_APPROVAL_DATE'] + '-01')
print(f'Train: {train_df.period.min().date()} to {train_df.period.max().date()} ({len(train_df):,} rows)')
print(f'Test:  {test_df.period.min().date()} to {test_df.period.max().date()} ({len(test_df):,} rows)')

/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 2021-01-01 to 2025-03-01 (150,000 rows)
Test:  2025-03-01 to 2026-07-01 (50,000 rows)


## Layer 1 — aggregate monthly market model

Only one aggregate feature is used for stocks (`stock_close_mean`), alongside aggregate COE variables. Individual stock columns are intentionally excluded. The final chronological 20% of observed training months is held out for validation.

In [2]:
TIME_FEATURES = [
    'time_index', 'coe_quota_premium_mean', 'coe_bids_mean',
    'stock_close_mean', 'stock_volume_mean',
]
monthly = (train_df.groupby('period', as_index=False)
    .agg(average_rent=(TARGET, 'mean'),
         coe_quota_premium_mean=('coe_quota_premium_mean', 'mean'),
         coe_bids_mean=('coe_bids_mean', 'mean'),
         stock_close_mean=('stock_close_mean', 'mean'),
         stock_volume_mean=('stock_volume_mean', 'mean'))
    .sort_values('period').reset_index(drop=True))
monthly['time_index'] = np.arange(len(monthly), dtype=float)
split_month = monthly['period'].iloc[int(len(monthly) * 0.80)]
time_train = monthly[monthly['period'] < split_month].copy()
time_valid = monthly[monthly['period'] >= split_month].copy()

def make_time_model():
    return Pipeline([
        ('scale', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('ridge', Ridge(alpha=10.0)),
    ])

# Fit only on pre-split monthly aggregates.
time_model = make_time_model()
time_model.fit(time_train[TIME_FEATURES], time_train['average_rent'])
time_valid['time_prediction'] = time_model.predict(time_valid[TIME_FEATURES])
time_rmse = mean_squared_error(time_valid['average_rent'], time_valid['time_prediction']) ** 0.5
print(f'Time-model validation months: {len(time_valid)}')
print(f'Time-model validation RMSE: {time_rmse:,.3f}')
display(time_valid[['period', 'average_rent', 'time_prediction']])

Time-model validation months: 11
Time-model validation RMSE: 174.253


,period,average_rent,time_prediction
40,2024-05-01,3095.840570,2944.079914
41,2024-06-01,3103.949186,3060.447658
42,2024-07-01,3169.898305,3106.836053
43,2024-08-01,3126.059301,3092.992826
44,2024-09-01,3108.928730,3004.778722
45,2024-10-01,3123.879459,3254.894839
46,2024-11-01,3149.154328,3084.989229
47,2024-12-01,3123.598757,3325.390754
48,2025-01-01,3125.447387,3412.017804
49,2025-02-01,3138.288551,3332.246834


## Layer 2 — time-independent structural model

The structural training table is formed from rows before the same chronological cutoff only. Each structural group is averaged over those earlier months, so validation-period rents cannot influence the group target.

In [3]:
STRUCTURAL_FEATURES = [
    'BLOCK', 'STREET', 'TOWN', 'FLAT_TYPE', 'FLAT_MODEL', 'FURNISHED',
    'FLOOR_AREA_SQM', 'LEASE_COMMENCE_DATE', 'LATITUDE', 'LONGITUDE',
    'MAX_FLOOR', 'YEAR_COMPLETED', 'mrt_dist_nearest_km',
    'mall_dist_nearest_km', 'school_dist_nearest_km',
]
GROUP_KEYS = ['BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'LEASE_COMMENCE_DATE']
NON_TIME_COLUMNS = {'RENT_APPROVAL_DATE', 'period', 'approval_year', 'approval_month', 'approval_date_num', 'lease_age', *TIME_FEATURES[1:]}
assert not (set(STRUCTURAL_FEATURES) & NON_TIME_COLUMNS)

structural_train_rows = train_df[train_df['period'] < split_month].copy()
structural_train = (structural_train_rows.groupby(GROUP_KEYS, dropna=False, as_index=False)[TARGET]
    .mean().rename(columns={TARGET: 'structural_rent'}))
# Recover static explanatory columns from the pre-split rows only.
static_columns = [c for c in STRUCTURAL_FEATURES if c not in GROUP_KEYS]
static_lookup = structural_train_rows[GROUP_KEYS + static_columns].drop_duplicates(GROUP_KEYS)
structural_train = structural_train.merge(static_lookup, on=GROUP_KEYS, how='left')

def prepare_structural(frame):
    result = frame[STRUCTURAL_FEATURES].copy()
    for column in result.select_dtypes(include=['object', 'string']).columns:
        result[column] = result[column].fillna('Unknown').astype('category')
    return result

categorical_structural = [c for c in STRUCTURAL_FEATURES if train_df[c].dtype == 'object']
X_struct_train = prepare_structural(structural_train)
y_struct_train = structural_train['structural_rent'].astype(float)
structural_model = LGBMRegressor(
    n_estimators=700, learning_rate=0.04, num_leaves=31, max_depth=-1,
    min_child_samples=30, subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1,
)
structural_model.fit(X_struct_train, y_struct_train, categorical_feature=categorical_structural)

valid_rows = train_df[train_df['period'] >= split_month].copy()
X_struct_valid = prepare_structural(valid_rows)
# Match validation category vocabularies to training vocabularies.
for column in categorical_structural:
    X_struct_valid[column] = X_struct_valid[column].cat.set_categories(X_struct_train[column].cat.categories)
valid_rows['structural_prediction'] = structural_model.predict(X_struct_valid)
print(f'Structural groups trained: {len(structural_train):,}')
print(f'Structural features: {len(STRUCTURAL_FEATURES)}; time features excluded: {sorted(NON_TIME_COLUMNS)}')

Structural groups trained: 37,178
Structural features: 15; time features excluded: ['RENT_APPROVAL_DATE', 'approval_date_num', 'approval_month', 'approval_year', 'coe_bids_mean', 'coe_quota_premium_mean', 'lease_age', 'period', 'stock_close_mean', 'stock_volume_mean']


## Combine layers and validate

The two absolute rent predictions are combined as a weighted average. The weight is tuned from earlier out-of-fold predictions only, then evaluated on a later untouched chronological fold.

In [4]:
from sklearn.model_selection import TimeSeriesSplit

def fit_fold_predictions(train_rows, valid_rows):
    # Every object in this function is fitted using train_rows only.
    train_monthly = (train_rows.groupby('period', as_index=False)
        .agg(average_rent=(TARGET, 'mean'),
             coe_quota_premium_mean=('coe_quota_premium_mean', 'mean'),
             coe_bids_mean=('coe_bids_mean', 'mean'),
             stock_close_mean=('stock_close_mean', 'mean'),
             stock_volume_mean=('stock_volume_mean', 'mean'))
        .sort_values('period').reset_index(drop=True))
    train_monthly['time_index'] = ((train_monthly['period'].dt.year - train_df['period'].min().year) * 12
        + train_monthly['period'].dt.month - train_df['period'].min().month).astype(float)
    valid_monthly = (valid_rows.groupby('period', as_index=False)
        .agg(coe_quota_premium_mean=('coe_quota_premium_mean', 'mean'),
             coe_bids_mean=('coe_bids_mean', 'mean'),
             stock_close_mean=('stock_close_mean', 'mean'),
             stock_volume_mean=('stock_volume_mean', 'mean')))
    valid_monthly['time_index'] = ((valid_monthly['period'].dt.year - train_df['period'].min().year) * 12
        + valid_monthly['period'].dt.month - train_df['period'].min().month).astype(float)
    fold_time_model = make_time_model().fit(train_monthly[TIME_FEATURES], train_monthly['average_rent'])
    valid_monthly['time_prediction'] = fold_time_model.predict(valid_monthly[TIME_FEATURES])
    time_predictions = valid_rows['period'].map(valid_monthly.set_index('period')['time_prediction'])

    grouped = (train_rows.groupby(GROUP_KEYS, dropna=False, as_index=False)[TARGET]
        .mean().rename(columns={TARGET: 'structural_rent'}))
    lookup = train_rows[GROUP_KEYS + static_columns].drop_duplicates(GROUP_KEYS)
    grouped = grouped.merge(lookup, on=GROUP_KEYS, how='left')
    X_fold = prepare_structural(grouped)
    X_valid_fold = prepare_structural(valid_rows)
    for column in categorical_structural:
        X_valid_fold[column] = X_valid_fold[column].cat.set_categories(X_fold[column].cat.categories)
    fold_model = LGBMRegressor(
        n_estimators=700, learning_rate=0.04, num_leaves=31,
        min_child_samples=30, subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1,
    )
    fold_model.fit(X_fold, grouped['structural_rent'].astype(float), categorical_feature=categorical_structural)
    structural_predictions = fold_model.predict(X_valid_fold)
    return pd.DataFrame({
        'period': valid_rows['period'].to_numpy(),
        'actual': valid_rows[TARGET].to_numpy(),
        'time_prediction': time_predictions.to_numpy(),
        'structural_prediction': structural_predictions,
    }, index=valid_rows.index)

periods = np.array(sorted(train_df['period'].unique()))
cv = TimeSeriesSplit(n_splits=5)
oof_parts = []
for fold_number, (train_idx, valid_idx) in enumerate(cv.split(periods), start=1):
    fold_train_periods = set(periods[train_idx])
    fold_valid_periods = set(periods[valid_idx])
    fold_train_rows = train_df[train_df['period'].isin(fold_train_periods)].copy()
    fold_valid_rows = train_df[train_df['period'].isin(fold_valid_periods)].copy()
    fold_predictions = fit_fold_predictions(fold_train_rows, fold_valid_rows)
    fold_predictions['fold'] = fold_number
    oof_parts.append(fold_predictions)
    print(f'Fold {fold_number}: train through {periods[train_idx[-1]].date()}, validate {len(fold_valid_rows):,} rows')
oof = pd.concat(oof_parts).sort_index()

# Tune only on folds 1-4; fold 5 remains untouched for the blend evaluation.
weight_tune = oof[oof['fold'] < oof['fold'].max()]
weight_holdout = oof[oof['fold'] == oof['fold'].max()]
def metrics(name, actual, prediction):
    return {'model': name, 'RMSE': mean_squared_error(actual, prediction) ** 0.5,
            'MAE': mean_absolute_error(actual, prediction), 'R2': r2_score(actual, prediction)}

def weight_objective(trial):
    time_weight = trial.suggest_float('time_weight', 0.0, 1.0)
    prediction = (time_weight * weight_tune['time_prediction']
        + (1.0 - time_weight) * weight_tune['structural_prediction'])
    return mean_squared_error(weight_tune['actual'], prediction) ** 0.5
weight_study = optuna.create_study(direction='minimize', study_name='exp_4_two_layer_blend',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
weight_study.optimize(weight_objective, n_trials=50, show_progress_bar=False)
blend_weight = weight_study.best_params['time_weight']

def weighted_prediction(frame, weight):
    return weight * frame['time_prediction'] + (1.0 - weight) * frame['structural_prediction']
weight_holdout = weight_holdout.copy()
weight_holdout['weighted_prediction'] = weighted_prediction(weight_holdout, blend_weight)
comparison = pd.DataFrame([
    metrics('Structural only', weight_holdout['actual'], weight_holdout['structural_prediction']),
    metrics('Time only', weight_holdout['actual'], weight_holdout['time_prediction']),
    metrics('Tuned weighted average', weight_holdout['actual'], weight_holdout['weighted_prediction']),
])
display(comparison.style.format({'RMSE': '{:,.3f}', 'MAE': '{:,.3f}', 'R2': '{:.4f}'}))
print(comparison.to_string(index=False, formatters={'RMSE': '{:,.3f}'.format, 'MAE': '{:,.3f}'.format, 'R2': '{:.4f}'.format}))
print(f'Tuned time-model weight: {blend_weight:.4f}; structural weight: {1.0 - blend_weight:.4f}')
blend_results_path = experiment_dir / 'exp_4_two_layer_blend_results.csv'
comparison.assign(time_weight=blend_weight, structural_weight=1.0 - blend_weight).to_csv(blend_results_path, index=False)
print(f'Saved blend validation results to {blend_results_path}')

Fold 1: train through 2021-11-01, validate 23,193 rows
Fold 2: train through 2022-07-01, validate 21,297 rows
Fold 3: train through 2023-03-01, validate 23,869 rows
Fold 4: train through 2023-11-01, validate 24,432 rows


[I 2026-09-14 16:48:57,160] A new study created in memory with name: exp_4_two_layer_blend
[I 2026-09-14 16:48:57,161] Trial 0 finished with value: 1009.3698703340021 and parameters: {'time_weight': 0.3745401188473625}. Best is trial 0 with value: 1009.3698703340021.
[I 2026-09-14 16:48:57,162] Trial 1 finished with value: 2129.4836446544023 and parameters: {'time_weight': 0.9507143064099162}. Best is trial 0 with value: 1009.3698703340021.
[I 2026-09-14 16:48:57,163] Trial 2 finished with value: 1674.3394919163732 and parameters: {'time_weight': 0.7319939418114051}. Best is trial 0 with value: 1009.3698703340021.
[I 2026-09-14 16:48:57,164] Trial 3 finished with value: 1409.3029875877585 and parameters: {'time_weight': 0.5986584841970366}. Best is trial 0 with value: 1009.3698703340021.
[I 2026-09-14 16:48:57,165] Trial 4 finished with value: 752.5870976226936 and parameters: {'time_weight': 0.15601864044243652}. Best is trial 4 with value: 752.5870976226936.
[I 2026-09-14 16:48:57,16

Fold 5: train through 2024-07-01, validate 22,739 rows


,model,RMSE,MAE,R2
0,Structural only,548.904,451.345,0.2750
1,Time only,674.200,515.875,-0.0937
2,Tuned weighted average,538.165,437.511,0.3031


                 model    RMSE     MAE      R2
       Structural only 548.904 451.345  0.2750
             Time only 674.200 515.875 -0.0937
Tuned weighted average 538.165 437.511  0.3031
Tuned time-model weight: 0.0607; structural weight: 0.9393
Saved blend validation results to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_4_two_layer_blend_results.csv


In [5]:
# Refit both layers on all training data for the final test prediction.
monthly_full = monthly.copy()
time_model_final = make_time_model().fit(monthly_full[TIME_FEATURES], monthly_full['average_rent'])
monthly_full['time_prediction'] = time_model_final.predict(monthly_full[TIME_FEATURES])
# Build test-month covariates without using any test rent target. The time
# index continues from the first training month for future extrapolation.
test_monthly = (test_df.groupby('period', as_index=False)
    .agg(coe_quota_premium_mean=('coe_quota_premium_mean', 'mean'),
         coe_bids_mean=('coe_bids_mean', 'mean'),
         stock_close_mean=('stock_close_mean', 'mean'),
         stock_volume_mean=('stock_volume_mean', 'mean')))
test_monthly['time_index'] = ((test_monthly['period'].dt.year - monthly['period'].min().year) * 12
    + test_monthly['period'].dt.month - monthly['period'].min().month).astype(float)
test_monthly['time_prediction'] = time_model_final.predict(test_monthly[TIME_FEATURES])

structural_train_full = (train_df.groupby(GROUP_KEYS, dropna=False, as_index=False)[TARGET]
    .mean().rename(columns={TARGET: 'structural_rent'}))
static_lookup_full = train_df[GROUP_KEYS + static_columns].drop_duplicates(GROUP_KEYS)
structural_train_full = structural_train_full.merge(static_lookup_full, on=GROUP_KEYS, how='left')
X_struct_full = prepare_structural(structural_train_full)
y_struct_full = structural_train_full['structural_rent'].astype(float)
structural_model_final = LGBMRegressor(
    n_estimators=700, learning_rate=0.04, num_leaves=31, max_depth=-1,
    min_child_samples=30, subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1,
)
structural_model_final.fit(X_struct_full, y_struct_full, categorical_feature=categorical_structural)

X_test_struct = prepare_structural(test_df)
for column in categorical_structural:
    X_test_struct[column] = X_test_struct[column].cat.set_categories(X_struct_full[column].cat.categories)
test_output = test_raw.copy()
test_output['structural_prediction'] = structural_model_final.predict(X_test_struct)
test_output['period'] = test_df['period'].to_numpy()
test_output['time_prediction'] = test_output['period'].map(
    test_monthly.set_index('period')['time_prediction']
)
test_output['PREDICTED_MONTHLY_RENT'] = (
    blend_weight * test_output['time_prediction']
    + (1.0 - blend_weight) * test_output['structural_prediction']
)
test_output['PREDICTED_MONTHLY_RENT'] = test_output['PREDICTED_MONTHLY_RENT'].clip(lower=0)
output_path = experiment_dir / 'exp_4_two_layer_predictions.csv'
test_output.drop(columns=['period', 'structural_prediction', 'time_prediction']).to_csv(output_path, index=False)
print(f'Saved predictions to {output_path}')
display(test_output[['RENT_APPROVAL_DATE', 'structural_prediction', 'time_prediction', 'PREDICTED_MONTHLY_RENT']].head())

Saved predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_4_two_layer_predictions.csv


,RENT_APPROVAL_DATE,structural_prediction,time_prediction,PREDICTED_MONTHLY_RENT
0,2025-03,2504.395095,3197.406285,2546.487817
1,2025-03,3240.186999,3197.406285,3237.588546
2,2025-03,2937.615404,3197.406285,2953.394811
3,2025-03,2882.342760,3197.406285,2901.479365
4,2025-03,3108.547142,3197.406285,3113.944346
